In [2]:
! pip install transformers torch accelerate


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# chatbot.py
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# MODEL_ID = "microsoft/Phi-4-mini-instruct"
MODEL_ID = r"C:\phi4-mini" 

print("Loading model... (first run downloads ~2.5 GB, then it's cached offline)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    # torch_dtype=torch.float16,   # halves memory usage
    torch_dtype=torch.float32,   # CPU needs float32
    device_map="cpu",            # uses GPU if available, else CPU
)
model.eval()
print("Model ready! Type 'quit' to exit.\n")

history = []

def chat(user_input):
    history.append({"role": "user", "content": user_input})

    # Build prompt using the model's chat template
    prompt = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (not the input prompt)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    history.append({"role": "assistant", "content": response})
    return response


while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "bye"):
        print("Bye!")
        break
    reply = chat(user_input)
    print(f"\nBot: {reply}\n")